In [1]:
import pandas as pd
import os

In [2]:
def read_signal_file(filepath, signal_type):
    with open(filepath, "r") as file:
        start_time = pd.to_datetime(file.readline().strip().split(",")[0])

        if signal_type != "tags":
            sample_rate = float(file.readline().strip().split(",")[0])

        if signal_type == "ACC":
            data = pd.read_csv(
                filepath, skiprows=2, delimiter=",", names=["x", "y", "z"], dtype=float
            )
            timestamps = pd.date_range(
                start=start_time, periods=len(data), freq=f"{1000/32:.0f}ms"
            )
            return pd.DataFrame(data.values, columns=["x", "y", "z"], index=timestamps)

        elif signal_type == "tags":
            df = pd.read_csv(filepath, header=None, names=["timestamp"])
            df["timestamp"] = pd.to_datetime(df["timestamp"])
            return df

        else:
            data = pd.read_csv(
                filepath,
                skiprows=2,
                header=None,
                names=[signal_type.lower()],
                dtype=float,
            )
            timestamps = pd.date_range(
                start=start_time, periods=len(data), freq=f"{1000/sample_rate:.0f}ms"
            )
            return pd.DataFrame(
                data.values, columns=[signal_type.lower()], index=timestamps
            )


def add_phase_labels(df, tags_df):
    tag_times = tags_df["timestamp"].tolist()
    df["phase"] = None

    phase_mappings = [
        (slice(None, tag_times[0]), "baseline"),
        (slice(tag_times[1], tag_times[2]), "tmct"),
        (slice(tag_times[2], tag_times[3]), "first rest"),
        (slice(tag_times[3], tag_times[4]), "real opinion"),
        (slice(tag_times[5], tag_times[6]), "opposite opinion"),
        (slice(tag_times[6], tag_times[7]), "second rest"),
        (slice(tag_times[7], tag_times[8]), "subtract test"),
    ]

    for time_slice, phase_name in phase_mappings:
        df.loc[time_slice, "phase"] = phase_name

    return df


def process_participant(participant_path):
    signals = {}
    for signal_type in ["ACC", "BVP", "EDA", "HR", "TEMP"]:
        filepath = os.path.join(participant_path, f"{signal_type}.csv")
        signals[signal_type] = read_signal_file(filepath, signal_type)

    start_times = [df.index[0] for df in signals.values()]
    end_times = [df.index[-1] for df in signals.values()]
    overall_start = max(start_times)
    overall_end = min(end_times)

    common_index = pd.date_range(start=overall_start, end=overall_end, freq="250ms")

    aligned_signals = {}
    for signal_type, df in signals.items():
        df = df[overall_start:overall_end]

        if signal_type == "ACC":
            resampled = pd.DataFrame()
            for col in ["x", "y", "z"]:
                temp = df[col].resample("250ms").mean().round(3)
                resampled[f"acc_{col}"] = temp
        elif signal_type == "HR":
            resampled = df.resample("250ms").mean()
            resampled = resampled.interpolate(method="cubic").round(2)
        elif signal_type == "TEMP":
            resampled = df.resample("250ms").mean().round(2)
        elif signal_type == "EDA":
            resampled = df.resample("250ms").mean().round(6)
        else:
            resampled = df.resample("250ms").mean().round(6)

        aligned_signals[signal_type] = resampled

    final_df = pd.DataFrame(index=common_index)
    final_df[["acc_x", "acc_y", "acc_z"]] = aligned_signals["ACC"]

    for signal_type in ["BVP", "EDA", "HR", "TEMP"]:
        final_df[signal_type.lower()] = aligned_signals[signal_type]

    final_df["relative_time"] = (
        (final_df.index - final_df.index[0]).total_seconds().round(2)
    )

    tags_df = read_signal_file(os.path.join(participant_path, "tags.csv"), "tags")
    final_df = add_phase_labels(final_df, tags_df)

    return final_df


def main(participant_ids, input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for participant in participant_ids:
        try:
            print(f"Processing participant {participant}...")
            participant_path = os.path.join(input_dir, participant)

            if not os.path.exists(participant_path):
                print(
                    f"Warning: Directory for participant {participant} not found. Skipping..."
                )
                continue

            participant_df = process_participant(participant_path)

            output_path = os.path.join(output_dir, f"{participant}.csv")
            participant_df.to_csv(output_path, index_label="timestamp")

            print(f"Successfully processed participant {participant}")
            print(f"Saved to: {output_path}")
            print(f"Number of rows: {len(participant_df)}")

        except Exception as e:
            print(f"Error processing participant {participant}: {str(e)}")
            continue

In [4]:
participant_ids = [f'f{str(i).zfill(2)}' for i in range(1, 19) if i not in [7, 14]]
input_dir = '../original/Wearable_Dataset/STRESS/'
output_dir = '../data'

main(participant_ids, input_dir, output_dir)

Processing participant f01...
Successfully processed participant f01
Saved to: ../data\f01.csv
Number of rows: 12865
Processing participant f02...
Successfully processed participant f02
Saved to: ../data\f02.csv
Number of rows: 11766
Processing participant f03...
Successfully processed participant f03
Saved to: ../data\f03.csv
Number of rows: 14228
Processing participant f04...
Successfully processed participant f04
Saved to: ../data\f04.csv
Number of rows: 14431
Processing participant f05...
Successfully processed participant f05
Saved to: ../data\f05.csv
Number of rows: 18183
Processing participant f06...
Successfully processed participant f06
Saved to: ../data\f06.csv
Number of rows: 12547
Processing participant f08...
Successfully processed participant f08
Saved to: ../data\f08.csv
Number of rows: 12582
Processing participant f09...
Successfully processed participant f09
Saved to: ../data\f09.csv
Number of rows: 12791
Processing participant f10...
Successfully processed participant